# MLP on MNIST

A multi-layer perceptron (MLP) is the simplest neural network architecture. This notebook trains a small MLP on MNIST handwritten digits and tracks loss and accuracy over epochs.

## Imports and data

We import PyTorch and torchvision, then download the MNIST dataset (28×28 grayscale images of handwritten digits). The data is split into 60 000 training and 10 000 test samples and wrapped in `DataLoader` objects for efficient batching.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = Path("../../data/raw")

train_ds = datasets.MNIST(data_dir, train=True, download=True, transform=transforms.ToTensor())
test_ds = datasets.MNIST(data_dir, train=False, download=True, transform=transforms.ToTensor())

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)

print(f"Train: {len(train_ds)} | Test: {len(test_ds)} | Device: {device}")

## Sample images

We display a few training images to get a visual sense of the data — each image is a single-channel 28×28 pixel grayscale digit.

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(12, 1.5))
for i, ax in enumerate(axes):
    img, label = train_ds[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.tight_layout()
plt.show()

## Define model

We build a simple MLP (Multi-Layer Perceptron) with one hidden layer of 128 neurons and ReLU activation. The input is flattened from 28×28 = 784 pixels to a vector, and the output layer has 10 neurons (one per digit class). We use Adam as optimizer and CrossEntropyLoss as the loss function.

In [ ]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

model

## Train and evaluate

We run 5 epochs of training. In each epoch, we iterate over all batches, compute the loss, backpropagate gradients, and update the weights. After each epoch we measure accuracy on the test set without computing gradients (`torch.no_grad`).

In [ ]:
history = {"epoch": [], "loss": [], "accuracy": []}

for epoch in range(1, 6):
    model.train()
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = loss_fn(model(images), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    correct = sum(
        (model(img.to(device)).argmax(1) == lab.to(device)).sum().item()
        for img, lab in test_loader
    )
    acc = correct / len(test_ds)
    avg_loss = total_loss / len(train_loader)

    history["epoch"].append(epoch)
    history["loss"].append(avg_loss)
    history["accuracy"].append(acc)
    print(f"Epoch {epoch}: loss={avg_loss:.4f}, accuracy={acc:.3f}")

## Training curves

We plot the training loss and test accuracy across epochs. A decreasing loss and increasing accuracy indicate the model is learning correctly.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(history["epoch"], history["loss"], marker="o")
ax1.set(xlabel="Epoch", ylabel="Loss", title="Training loss")
ax2.plot(history["epoch"], history["accuracy"], marker="o", color="green")
ax2.set(xlabel="Epoch", ylabel="Accuracy", title="Test accuracy")
plt.tight_layout()
plt.show()

## Practical note

An MLP is the simplest deep learning model and a natural first baseline. It treats images as flat vectors, ignoring spatial structure. For vision tasks a CNN usually works better, but the MLP is great for understanding training loops and backpropagation basics.